# 📊 VQA-E — Monitor & Manage Training: Model C & D

**Mục đích**: Theo dõi training progress, validate checkpoints, chạy evaluation trung gian cho Model C và D đang train.

| Model | CNN | Attention | Coverage | Mô tả |
|-------|-----|-----------|----------|--------|
| **C** | SimpleCNN Spatial (49 regions) | ✅ Dual Attention | Phase 2+ | Scratch CNN + Attention |
| **D** | ResNet101 Spatial (49 regions) | ✅ Dual Attention | Phase 2+ | Full model (best expected) |

**Training Plan**: Phase 1 (15ep, frozen) → Phase 2 (10ep, unfreeze+coverage) → Phase 3 (5ep, SS)

> 💡 Notebook này chạy **độc lập** với training notebook — chỉ đọc checkpoint files và history JSON.

---
## 1. Environment Setup & GPU Check

In [ ]:
import os, sys, json, torch, random
import numpy as np

# ── Set working directory to project root ────────────────────────────────────
PROJECT_ROOT = "/home/mayxin/workspace/DeepLearning/new_vqa"
os.chdir(PROJECT_ROOT)
sys.path.insert(0, 'src')
print(f"Working directory: {os.getcwd()}")

# ── PyTorch & CUDA info ───────────────────────────────────────────────────────
print(f"\nPyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    cap = torch.cuda.get_device_capability()
    vram_gb = gpu.total_memory / 1e9
    bf16_ok = cap[0] >= 8
    print(f"GPU             : {gpu.name}")
    print(f"VRAM            : {vram_gb:.1f} GB")
    print(f"Compute cap.    : {cap[0]}.{cap[1]}")
    print(f"BF16 support    : {'✅ YES (Ampere+)' if bf16_ok else '❌ NO — will use FP16'}")
    print(f"TF32 matmul     : {'✅ Enabled' if cap[0] >= 8 else '❌ Not available'}")
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print("⚠️ WARNING: CUDA not available")

---
## 2. Training Configuration — Model C & D

Hyperparameters cụ thể cho Model C (SimpleCNN Spatial) và Model D (ResNet101 Spatial).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TRAINING CONFIG — Model C & D (RTX 3060, 12 GB VRAM, BF16)
# ══════════════════════════════════════════════════════════════════════════════

CONFIG = {
    'C': {
        'name': 'SimpleCNN Spatial + Dual Attention',
        'phase1': {'epochs': 15, 'lr': 5e-4, 'batch': 96, 'accum': 1, 'vram': '~5.9 GB'},
        'phase2': {'epochs': 10, 'lr': 3e-4, 'batch': 96, 'accum': 1, 'vram': '~5.9 GB',
                   'coverage': True, 'coverage_lambda': 0.5},
        'phase3': {'epochs': 5,  'lr': 1e-4, 'batch': 96, 'accum': 1, 'vram': '~5.9 GB',
                   'coverage': True, 'scheduled_sampling': True, 'ss_k': 10},
    },
    'D': {
        'name': 'ResNet101 Spatial + Dual Attention',
        'phase1': {'epochs': 15, 'lr': 5e-4, 'batch': 128, 'accum': 1, 'vram': '~2.4 GB'},
        'phase2': {'epochs': 10, 'lr': 3e-4, 'batch': 96,  'accum': 1, 'vram': '~5.3 GB',
                   'coverage': True, 'coverage_lambda': 0.5, 'finetune_cnn': True, 'cnn_lr_factor': 0.1},
        'phase3': {'epochs': 5,  'lr': 1e-4, 'batch': 96,  'accum': 1, 'vram': '~5.3 GB',
                   'coverage': True, 'scheduled_sampling': True, 'ss_k': 10},
    },
}

COMMON = {
    'weight_decay': 1e-4, 'dropout': 0.3, 'grad_clip': 5.0,
    'label_smoothing': 0.1, 'warmup_epochs': 3, 'early_stopping': 10,
    'num_workers': 8, 'lr_schedule': 'CosineAnnealing',
}

print("=" * 70)
print("  TRAINING CONFIG — Model C & D")
print("=" * 70)
for m in ('C', 'D'):
    c = CONFIG[m]
    print(f"\n  Model {m}: {c['name']}")
    print(f"  {'─'*60}")
    for phase in ('phase1', 'phase2', 'phase3'):
        p = c[phase]
        eff = p['batch'] * p['accum']
        extras = []
        if p.get('coverage'):      extras.append(f"coverage(λ={p['coverage_lambda']})")
        if p.get('finetune_cnn'):   extras.append(f"unfreeze(×{p['cnn_lr_factor']})")
        if p.get('scheduled_sampling'): extras.append(f"SS(k={p['ss_k']})")
        ext_str = ' | '.join(extras) if extras else '—'
        print(f"    {phase}: {p['epochs']}ep | LR={p['lr']:.0e} | BS={p['batch']}×{p['accum']}={eff} | VRAM {p['vram']} | {ext_str}")

print(f"\n  Common: wd={COMMON['weight_decay']}, grad_clip={COMMON['grad_clip']}, "
      f"dropout={COMMON['dropout']}, LS={COMMON['label_smoothing']}")
print(f"  Schedule: {COMMON['lr_schedule']} | Warmup={COMMON['warmup_epochs']}ep (Phase 1 only)")
print(f"  Total: 15 + 10 + 5 = 30 epochs per model")

---
## 3. Current Training Progress — Model C & D

Đọc history files để xác định epoch hiện tại, phase, best val loss.

In [ ]:
# ── Check Current Training Progress ──────────────────────────────────────────
import json, os
from datetime import datetime

def get_progress(model_letter):
    """Read training history and return progress info."""
    hp = f"checkpoints/history_model_{model_letter.lower()}.json"
    resume_path = f"checkpoints/model_{model_letter.lower()}_resume.pth"
    best_path = f"checkpoints/model_{model_letter.lower()}_best.pth"

    info = {'model': model_letter, 'trained': False}
    if not os.path.exists(hp):
        return info

    h = json.load(open(hp))
    n_epochs = len(h['train_loss'])
    info['trained'] = True
    info['epochs'] = n_epochs
    info['phase'] = 'Phase 1' if n_epochs <= 15 else ('Phase 2' if n_epochs <= 25 else 'Phase 3')
    info['train_loss'] = h['train_loss'][-1] if h['train_loss'] else None
    info['val_loss'] = h['val_loss'][-1] if h['val_loss'] else None
    info['best_val'] = min(h['val_loss']) if h['val_loss'] else None
    info['best_epoch'] = h['val_loss'].index(info['best_val']) + 1 if h['val_loss'] else None
    info['has_resume'] = os.path.exists(resume_path)
    info['has_best'] = os.path.exists(best_path)
    info['history'] = h

    # LR info (if available)
    if 'lr' in h and h['lr']:
        info['current_lr'] = h['lr'][-1]

    # BLEU info (if available)
    if 'val_bleu4' in h and h['val_bleu4']:
        info['best_bleu4'] = max(h['val_bleu4'])
        info['last_bleu4'] = h['val_bleu4'][-1]

    # Checkpoint file timestamps
    if os.path.exists(resume_path):
        info['resume_time'] = datetime.fromtimestamp(os.path.getmtime(resume_path))

    return info

# ── Display Progress Table ───────────────────────────────────────────────────
print("╔═══════════════════════════════════════════════════════════════════════╗")
print("║              TRAINING PROGRESS — Model C & D                       ║")
print("╠═══════════════════════════════════════════════════════════════════════╣")

for m in ('C', 'D'):
    info = get_progress(m)
    if not info['trained']:
        print(f"║  Model {m}: ❌ NOT TRAINED YET                                       ║")
        continue

    phase_pct = {
        'Phase 1': f"{info['epochs']}/15",
        'Phase 2': f"{info['epochs']}/25",
        'Phase 3': f"{info['epochs']}/30",
    }[info['phase']]

    overall_pct = info['epochs'] / 30 * 100

    print(f"║  Model {m}: {CONFIG[m]['name']:<45}  ║")
    print(f"║    Epochs     : {info['epochs']:>3}/30  ({overall_pct:5.1f}%)  [{info['phase']} — {phase_pct}]        ║")
    print(f"║    Train Loss : {info['train_loss']:>10.4f}                                       ║")
    print(f"║    Val Loss   : {info['val_loss']:>10.4f}  (best: {info['best_val']:.4f} @ epoch {info['best_epoch']})    ║")

    if 'best_bleu4' in info:
        print(f"║    Val BLEU-4 : {info['last_bleu4']:>10.4f}  (best: {info['best_bleu4']:.4f})                  ║")
    if 'current_lr' in info:
        print(f"║    Current LR : {info['current_lr']:>10.2e}                                       ║")

    resume_icon = '✅' if info['has_resume'] else '❌'
    best_icon = '✅' if info['has_best'] else '❌'
    print(f"║    Checkpoints: resume {resume_icon}  best {best_icon}                                ║")

    if 'resume_time' in info:
        print(f"║    Last update: {info['resume_time'].strftime('%Y-%m-%d %H:%M:%S')}                           ║")

    print("║                                                                       ║")

print("╚═══════════════════════════════════════════════════════════════════════╝")

# Store for later use
progress_c = get_progress('C')
progress_d = get_progress('D')

---
## 4. Monitor Model C — Training Curves

Train loss, val loss, val BLEU-4 cho Model C. Annotate current epoch, LR, warmup status.

In [ ]:
# ── Model C Training Curves ──────────────────────────────────────────────────
%matplotlib inline
import matplotlib.pyplot as plt

def plot_model_curves(model_letter, color='#2ca02c'):
    """Plot train/val loss and BLEU-4 curves for a single model."""
    hp = f"checkpoints/history_model_{model_letter.lower()}.json"
    if not os.path.exists(hp):
        print(f"⚠ history_model_{model_letter.lower()}.json not found — Model {model_letter} chưa train")
        return

    h = json.load(open(hp))
    n = len(h['train_loss'])
    epochs = list(range(1, n + 1))
    has_bleu = 'val_bleu4' in h and len(h['val_bleu4']) > 0
    has_lr = 'lr' in h and len(h['lr']) > 0

    ncols = 2 + int(has_bleu) + int(has_lr)
    fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 4))

    # Train loss
    axes[0].plot(epochs, h['train_loss'], color=color, marker='o', ms=3, lw=1.5)
    axes[0].set_title(f'Model {model_letter} — Train Loss', fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].grid(alpha=0.3)

    # Val loss
    axes[1].plot(epochs, h['val_loss'], color=color, marker='s', ms=3, lw=1.5, ls='--')
    best_val = min(h['val_loss'])
    best_ep = h['val_loss'].index(best_val) + 1
    axes[1].axhline(best_val, color='red', ls=':', alpha=0.5, lw=1)
    axes[1].annotate(f'Best: {best_val:.4f} @ ep{best_ep}', xy=(best_ep, best_val),
                     fontsize=8, color='red', ha='left')
    axes[1].set_title(f'Model {model_letter} — Val Loss', fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].grid(alpha=0.3)

    idx = 2
    # BLEU-4
    if has_bleu:
        axes[idx].plot(epochs[:len(h['val_bleu4'])], h['val_bleu4'], color=color,
                       marker='^', ms=3, lw=1.5)
        best_bleu = max(h['val_bleu4'])
        axes[idx].axhline(best_bleu, color='red', ls=':', alpha=0.5, lw=1)
        axes[idx].set_title(f'Model {model_letter} — Val BLEU-4', fontweight='bold')
        axes[idx].set_xlabel('Epoch'); axes[idx].set_ylabel('BLEU-4')
        axes[idx].grid(alpha=0.3)
        idx += 1

    # Learning rate
    if has_lr:
        axes[idx].plot(epochs[:len(h['lr'])], h['lr'], color='gray', marker='d', ms=3, lw=1.5)
        axes[idx].set_title(f'Model {model_letter} — Learning Rate', fontweight='bold')
        axes[idx].set_xlabel('Epoch'); axes[idx].set_ylabel('LR')
        axes[idx].set_yscale('log')
        axes[idx].grid(alpha=0.3)
        # Warmup annotation
        if n >= 3:
            axes[idx].axvspan(0.5, 3.5, alpha=0.1, color='orange', label='Warmup')
            axes[idx].legend(fontsize=8)

    # Phase boundaries
    for ax in axes:
        for pe in [15, 25]:
            if pe <= n:
                ax.axvline(pe, color='gray', ls=':', alpha=0.5, lw=1)

    plt.suptitle(f"Model {model_letter} — Epoch {n}/30", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Last 5 epochs table
    print(f"\n  Last {min(5, n)} epochs:")
    print(f"  {'Epoch':>6} {'Train Loss':>12} {'Val Loss':>12}", end='')
    if has_bleu:
        print(f" {'BLEU-4':>10}", end='')
    if has_lr:
        print(f" {'LR':>12}", end='')
    print()
    print(f"  {'─'*50}")
    for i in range(max(0, n - 5), n):
        ep = i + 1
        tl = h['train_loss'][i]
        vl = h['val_loss'][i]
        line = f"  {ep:>6} {tl:>12.4f} {vl:>12.4f}"
        if has_bleu and i < len(h['val_bleu4']):
            line += f" {h['val_bleu4'][i]:>10.4f}"
        if has_lr and i < len(h['lr']):
            line += f" {h['lr'][i]:>12.2e}"
        # Mark best
        if vl == best_val:
            line += "  ★ best"
        print(line)

plot_model_curves('C', color='#2ca02c')

---
## 5. Monitor Model D — Training Curves

Train loss, val loss, val BLEU-4 cho Model D (ResNet101 Spatial + Dual Attention).

In [ ]:
# ── Model D Training Curves ──────────────────────────────────────────────────
plot_model_curves('D', color='#d62728')

---
## 6. Live Loss Tracking — Model C vs D (Overlay)

So sánh trực tiếp train/val loss giữa C và D trên cùng 1 plot.

In [ ]:
# ── Model C vs D — Overlay Comparison ────────────────────────────────────────
%matplotlib inline
import matplotlib.pyplot as plt
import json, os

COLORS = {'C': '#2ca02c', 'D': '#d62728'}
LABELS = {'C': 'C: SimpleCNN+DualAttn', 'D': 'D: ResNet101+DualAttn'}

histories = {}
for m in ('C', 'D'):
    hp = f"checkpoints/history_model_{m.lower()}.json"
    if os.path.exists(hp):
        histories[m] = json.load(open(hp))

if not histories:
    print("⚠ No history files found for C or D")
else:
    has_bleu = any('val_bleu4' in h and h['val_bleu4'] for h in histories.values())
    ncols = 2 + int(has_bleu)
    fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 5))

    for m, h in histories.items():
        n = len(h['train_loss'])
        epochs = list(range(1, n + 1))

        axes[0].plot(epochs, h['train_loss'], color=COLORS[m], label=LABELS[m],
                     marker='o', ms=3, lw=1.5)
        axes[1].plot(epochs, h['val_loss'], color=COLORS[m], label=LABELS[m],
                     marker='s', ms=3, lw=1.5, ls='--')

        if has_bleu and 'val_bleu4' in h and h['val_bleu4']:
            axes[2].plot(epochs[:len(h['val_bleu4'])], h['val_bleu4'],
                         color=COLORS[m], label=LABELS[m], marker='^', ms=3, lw=1.5)

    titles = ['Train Loss', 'Val Loss'] + (['Val BLEU-4'] if has_bleu else [])
    ylabels = ['Loss', 'Loss'] + (['BLEU-4'] if has_bleu else [])
    for i, ax in enumerate(axes):
        ax.set_title(titles[i], fontsize=12, fontweight='bold')
        ax.set_xlabel('Epoch'); ax.set_ylabel(ylabels[i])
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)
        for pe in [15, 25]:
            ax.axvline(pe, color='gray', ls=':', alpha=0.4, lw=1)

    fig.suptitle('Model C vs D — Training Comparison', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # ── Side-by-side metrics table ───────────────────────────────────────────
    print(f"\n  {'Metric':<20}", end='')
    for m in ('C', 'D'):
        if m in histories:
            print(f" {'Model '+m:>12}", end='')
    print(f" {'Gap':>10}")
    print(f"  {'─'*55}")

    metrics = [
        ('Epochs', lambda h: len(h['train_loss'])),
        ('Train Loss', lambda h: h['train_loss'][-1]),
        ('Val Loss', lambda h: h['val_loss'][-1]),
        ('Best Val Loss', lambda h: min(h['val_loss'])),
        ('Best Epoch', lambda h: h['val_loss'].index(min(h['val_loss'])) + 1),
    ]

    for name, fn in metrics:
        vals = {}
        print(f"  {name:<20}", end='')
        for m in ('C', 'D'):
            if m in histories:
                v = fn(histories[m])
                vals[m] = v
                if isinstance(v, float):
                    print(f" {v:>12.4f}", end='')
                else:
                    print(f" {v:>12}", end='')
        if len(vals) == 2 and all(isinstance(v, (int, float)) for v in vals.values()):
            gap = vals['D'] - vals['C']
            print(f" {gap:>+10.4f}", end='')
        print()

    # Which model is converging faster?
    if len(histories) == 2:
        c_vl = histories['C']['val_loss'][-1]
        d_vl = histories['D']['val_loss'][-1]
        winner = 'D' if d_vl < c_vl else 'C'
        print(f"\n  → Model {winner} has lower val loss ({min(c_vl, d_vl):.4f} vs {max(c_vl, d_vl):.4f})")

---
## 7. VRAM & GPU Utilization Monitoring

Check GPU memory, temperature, power draw. So sánh với expected VRAM profile.